# 04 — Machine Learning Volatility

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
DIAG = ROOT / "results" / "diagnostics"

for p in [TABLES, DIAG]:
    p.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 21, 63]
TEST = pd.Timestamp("2025-01-01")
EPS = 1e-10
RS = 42

FOLDS = [
    ("2023", pd.Timestamp("2023-01-01"), pd.Timestamp("2023-12-31")),
    ("2024", pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")),
]

groups = pd.read_csv(TABLES / "02_feature_group_definition.csv")

def qlike(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ratio = np.maximum(y**2, EPS) / np.maximum(p**2, EPS)
    return float(np.mean(ratio - np.log(ratio) - 1))

def metrics(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    e = p - y
    return {
        "N": len(y),
        "MAE": float(np.mean(np.abs(e))),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "QLIKE": qlike(y, p),
        "Correlation": float(np.corrcoef(y, p)[0, 1]) if np.std(p) > 0 else np.nan,
        "Bias": float(np.mean(e)),
        "Actual_Std": float(np.std(y, ddof=1)) if len(y) > 1 else np.nan,
        "Prediction_Std": float(np.std(p, ddof=1)) if len(p) > 1 else np.nan,
    }

def candidate_models():
    out = {}

    for alpha in [0.01, 0.1, 1, 10, 100]:
        out[f"Ridge_a{alpha}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("model", Ridge(alpha=alpha)),
        ])

    for leaf in [3, 8, 15]:
        out[f"RF_leaf{leaf}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=400,
                min_samples_leaf=leaf,
                max_features=0.7,
                n_jobs=-1,
                random_state=RS,
            )),
        ])

    for leaf in [10, 20, 40]:
        out[f"HGB_leaf{leaf}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.05,
                min_samples_leaf=leaf,
                random_state=RS,
            )),
        ])

    try:
        from xgboost import XGBRegressor
        for depth in [2, 3]:
            out[f"XGB_d{depth}"] = Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("model", XGBRegressor(
                    n_estimators=400,
                    max_depth=depth,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    n_jobs=-1,
                    random_state=RS,
                    objective="reg:squarederror",
                )),
            ])
    except ImportError:
        pass

    return out

def feature_lists(d, h):
    target = f"Target_RV_{h}"
    available = [
        c for c in d.columns
        if c not in ["Date", "Sector_Return", target]
        and not c.startswith("Target_RV_")
    ]

    out = {}
    current = []

    for _, row in groups.iterrows():
        added = [] if pd.isna(row["Added"]) else [
            x.strip() for x in str(row["Added"]).split(",") if x.strip()
        ]
        current = list(dict.fromkeys(
            current + [x for x in added if x in available]
        ))
        out[row["Feature_Set"]] = current.copy()

    return out

def load_horizon(h):
    d = pd.read_parquet(PROCESSED / f"volatility_features_h{h}.parquet").copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d = d.sort_values("Date").reset_index(drop=True)
    numeric = d.select_dtypes(include=[np.number]).columns
    d[numeric] = d[numeric].replace([np.inf, -np.inf], np.nan)
    return d

## 1. Purged validation and locked model selection

In [2]:
validation_rows = []
validation_predictions = []
raw_floor_rows = []

for h in HORIZONS:
    d = load_horizon(h)
    y = f"Target_RV_{h}"

    for feature_set, features in feature_lists(d, h).items():
        if not features:
            continue

        for transform in ["raw", "log"]:
            for model_name in candidate_models().keys():
                for fold, start, end in FOLDS:

                    val = d.loc[
                        d["Date"].between(start, end)
                    ].dropna(subset=[y]).copy()

                    if val.empty:
                        continue

                    first = val.index.min()
                    train_end = max(0, first - h)

                    tr = d.iloc[:train_end].dropna(subset=[y]).copy()

                    if len(tr) < 252:
                        continue

                    model = candidate_models()[model_name]

                    y_train = tr[y].astype(float).to_numpy()
                    if transform == "log":
                        y_train = np.log(np.maximum(y_train, 1e-8))

                    model.fit(tr[features], y_train)
                    raw_pred = np.asarray(model.predict(val[features]), dtype=float)

                    if transform == "log":
                        pred = np.exp(raw_pred)
                        nonpositive_raw = 0
                    else:
                        nonpositive_raw = int((raw_pred <= 0).sum())
                        pred = raw_pred.copy()

                    # QLIKE requires positive volatility. Flooring is evaluation-only.
                    pred = np.maximum(pred, 1e-8)

                    row = {
                        "Horizon": h,
                        "Feature_Set": feature_set,
                        "Transform": transform,
                        "Model": model_name,
                        "Fold": fold,
                        "N_Features": len(features),
                        "Nonpositive_Raw_Predictions": nonpositive_raw,
                        **metrics(val[y].to_numpy(), pred),
                    }
                    validation_rows.append(row)

                    if nonpositive_raw:
                        raw_floor_rows.append({
                            "Horizon": h,
                            "Feature_Set": feature_set,
                            "Transform": transform,
                            "Model": model_name,
                            "Fold": fold,
                            "Nonpositive_Raw_Predictions": nonpositive_raw,
                            "N_Validation": len(val),
                        })

                    # Save prediction rows for audit/reproducibility.
                    validation_predictions.extend([
                        {
                            "Date": dte,
                            "Horizon": h,
                            "Feature_Set": feature_set,
                            "Transform": transform,
                            "Model": model_name,
                            "Fold": fold,
                            "Actual": float(actual),
                            "Prediction": float(p),
                        }
                        for dte, actual, p in zip(val["Date"], val[y], pred)
                    ])

valres = pd.DataFrame(validation_rows)
valres.to_csv(TABLES / "04_all_validation_results.csv", index=False)
pd.DataFrame(validation_predictions).to_csv(
    TABLES / "04_validation_predictions.csv", index=False
)

floor_audit = pd.DataFrame(raw_floor_rows)
floor_audit.to_csv(DIAG / "04_nonpositive_raw_prediction_audit.csv", index=False)

summary = (
    valres.groupby(
        ["Horizon", "Feature_Set", "Transform", "Model"],
        as_index=False
    )
    .agg(
        Mean_Validation_QLIKE=("QLIKE", "mean"),
        Validation_QLIKE_SD=("QLIKE", "std"),
        Mean_Validation_MAE=("MAE", "mean"),
        Mean_Validation_RMSE=("RMSE", "mean"),
        Mean_Validation_Correlation=("Correlation", "mean"),
        Total_Nonpositive_Raw_Predictions=("Nonpositive_Raw_Predictions", "sum"),
        N_Folds=("Fold", "nunique"),
    )
)

summary.to_csv(TABLES / "04_feature_stage_validation_summary.csv", index=False)

# Selection requires both planned validation folds.
eligible = summary.loc[summary["N_Folds"].eq(len(FOLDS))].copy()

selected = (
    eligible.sort_values(["Horizon", "Mean_Validation_QLIKE"])
    .groupby("Horizon", as_index=False)
    .first()
)

selected.to_csv(TABLES / "04_selected_ml_by_horizon.csv", index=False)

print("Locked selected ML configuration by horizon:")
display(selected)
print("\nRaw non-positive prediction audit:")
display(floor_audit.head(30))

Locked selected ML configuration by horizon:


,Horizon,Feature_Set,Transform,Model,Mean_Validation_QLIKE,Validation_QLIKE_SD,Mean_Validation_MAE,Mean_Validation_RMSE,Mean_Validation_Correlation,Total_Nonpositive_Raw_Predictions,N_Folds
0,5,F4_Market_Commodities,log,Ridge_a10,0.249683,0.019963,0.050458,0.062488,0.092123,0,2
1,21,F5_Plus_Macro,log,Ridge_a100,0.088589,0.053347,0.027826,0.035755,-0.041719,0,2
2,63,F5_Plus_Macro,raw,XGB_d2,0.078153,0.071375,0.033771,0.038660,0.276665,0,2



Raw non-positive prediction audit:


,Horizon,Feature_Set,Transform,Model,Fold,Nonpositive_Raw_Predictions,N_Validation
0,5,F5_Plus_Macro,raw,Ridge_a0.01,2023,150,252
1,5,F5_Plus_Macro,raw,Ridge_a0.1,2023,150,252
2,5,F5_Plus_Macro,raw,Ridge_a1,2023,152,252
3,5,F5_Plus_Macro,raw,Ridge_a10,2023,142,252
4,5,F5_Plus_Macro,raw,Ridge_a100,2023,58,252
5,21,F5_Plus_Macro,raw,Ridge_a0.01,2023,252,252
6,21,F5_Plus_Macro,raw,Ridge_a0.1,2023,252,252
7,21,F5_Plus_Macro,raw,Ridge_a1,2023,252,252
8,21,F5_Plus_Macro,raw,Ridge_a10,2023,252,252
9,21,F5_Plus_Macro,raw,Ridge_a100,2023,247,252


## 2. Feature-stage comparison — best configuration within each information set

In [3]:
# This is NOT a same-model causal ablation.
# Each stage reports the best model + transform available within that stage.

idx = (
    eligible.groupby(["Horizon", "Feature_Set"])["Mean_Validation_QLIKE"]
    .idxmin()
)

stage = eligible.loc[idx, [
    "Horizon", "Feature_Set", "Transform", "Model",
    "Mean_Validation_QLIKE", "Validation_QLIKE_SD",
    "Mean_Validation_MAE", "Mean_Validation_RMSE",
    "Mean_Validation_Correlation",
    "Total_Nonpositive_Raw_Predictions",
    "N_Folds"
]].copy()

order = groups["Feature_Set"].tolist()
stage["Feature_Set"] = pd.Categorical(
    stage["Feature_Set"],
    categories=order,
    ordered=True
)
stage = stage.sort_values(["Horizon", "Feature_Set"])

stage["Previous_QLIKE"] = (
    stage.groupby("Horizon")["Mean_Validation_QLIKE"]
    .shift()
)

stage["Incremental_Improvement_Pct"] = (
    100
    * (stage["Previous_QLIKE"] - stage["Mean_Validation_QLIKE"])
    / stage["Previous_QLIKE"]
)

stage.to_csv(
    TABLES / "04_feature_stage_incremental_contribution.csv",
    index=False
)

macro_stage = stage.loc[
    stage["Feature_Set"].astype(str).eq("F5_Plus_Macro")
].copy()

macro_stage.to_csv(
    TABLES / "04_macro_contribution.csv",
    index=False
)

display(stage)

,Horizon,Feature_Set,Transform,Model,Mean_Validation_QLIKE,Validation_QLIKE_SD,Mean_Validation_MAE,Mean_Validation_RMSE,Mean_Validation_Correlation,Total_Nonpositive_Raw_Predictions,N_Folds,Previous_QLIKE,Incremental_Improvement_Pct
10,5,F1_Volatility,log,Ridge_a100,0.285671,0.046014,0.060200,0.071555,0.004095,0,2,NaN,NaN
32,5,F2_Returns_Downside,log,Ridge_a0.01,0.275097,0.037781,0.057276,0.069396,0.009951,0,2,0.285671,3.701354
60,5,F3_Range_Liquidity,log,Ridge_a1,0.251631,0.039039,0.052099,0.063855,0.055860,0,2,0.275097,8.530119
87,5,F4_Market_Commodities,log,Ridge_a10,0.249683,0.019963,0.050458,0.062488,0.092123,0,2,0.251631,0.774012
114,5,F5_Plus_Macro,log,Ridge_a100,0.272524,0.032168,0.050204,0.062302,-0.020201,0,2,0.249683,-9.147721
141,21,F1_Volatility,log,XGB_d2,0.136238,0.102031,0.045839,0.052775,-0.165176,0,2,NaN,NaN
167,21,F2_Returns_Downside,log,XGB_d2,0.133644,0.105533,0.046215,0.052985,-0.093242,0,2,0.136238,1.903866
205,21,F3_Range_Liquidity,raw,Ridge_a100,0.116536,0.059368,0.041469,0.050271,0.108501,0,2,0.133644,12.800987
215,21,F4_Market_Commodities,log,Ridge_a0.1,0.091130,0.058361,0.033619,0.042754,0.109010,0,2,0.116536,21.800894
244,21,F5_Plus_Macro,log,Ridge_a100,0.088589,0.053347,0.027826,0.035755,-0.041719,0,2,0.091130,2.788443


## 3. Untouched 2025+ final test

In [4]:
test_metric_rows = []
test_prediction_rows = []

for _, s in selected.iterrows():
    h = int(s["Horizon"])
    d = load_horizon(h)
    y = f"Target_RV_{h}"
    features = feature_lists(d, h)[s["Feature_Set"]]

    test = d.loc[d["Date"] >= TEST].dropna(subset=[y]).copy()
    if test.empty:
        continue

    first = test.index.min()
    train_end = max(0, first - h)
    tr = d.iloc[:train_end].dropna(subset=[y]).copy()

    if len(tr) < 252:
        raise RuntimeError(f"Insufficient training rows for h={h}")

    model = candidate_models()[s["Model"]]

    y_train = tr[y].astype(float).to_numpy()
    if s["Transform"] == "log":
        y_train = np.log(np.maximum(y_train, 1e-8))

    model.fit(tr[features], y_train)
    raw_pred = np.asarray(model.predict(test[features]), dtype=float)

    if s["Transform"] == "log":
        pred = np.exp(raw_pred)
        nonpositive_raw = 0
    else:
        nonpositive_raw = int((raw_pred <= 0).sum())
        pred = raw_pred.copy()

    pred = np.maximum(pred, 1e-8)

    valid = np.isfinite(pred) & np.isfinite(test[y].astype(float).to_numpy())
    test_valid = test.loc[valid].copy()
    pred_valid = pred[valid]

    result = {
        "Horizon": h,
        "Feature_Set": s["Feature_Set"],
        "Transform": s["Transform"],
        "Selected_Model": s["Model"],
        "Model": "Selected_ML",
        "N_Features": len(features),
        "Nonpositive_Raw_Predictions": nonpositive_raw,
        **metrics(test_valid[y].to_numpy(), pred_valid),
    }
    result["Variability_Ratio"] = (
        result["Prediction_Std"] / result["Actual_Std"]
        if result["Actual_Std"] > 0 else np.nan
    )
    test_metric_rows.append(result)

    test_prediction_rows.extend([
        {
            "Date": dte,
            "Horizon": h,
            "Feature_Set": s["Feature_Set"],
            "Transform": s["Transform"],
            "Selected_Model": s["Model"],
            "Model": "Selected_ML",
            "Actual": float(actual),
            "Prediction": float(p),
        }
        for dte, actual, p in zip(
            test_valid["Date"], test_valid[y], pred_valid
        )
    ])

ml_test_metrics = pd.DataFrame(test_metric_rows)
ml_test_predictions = pd.DataFrame(test_prediction_rows)

ml_test_metrics.to_csv(TABLES / "04_selected_ml_test_metrics.csv", index=False)
ml_test_predictions.to_csv(TABLES / "04_selected_ml_test_predictions.csv", index=False)

display(ml_test_metrics)

,Horizon,Feature_Set,Transform,Selected_Model,Model,N_Features,Nonpositive_Raw_Predictions,N,MAE,RMSE,QLIKE,Correlation,Bias,Actual_Std,Prediction_Std,Variability_Ratio
0,5,F4_Market_Commodities,log,Ridge_a10,Selected_ML,42,0,430,0.086031,0.113751,0.663734,0.306993,-0.046240,0.109326,0.032799,0.300009
1,21,F5_Plus_Macro,log,Ridge_a100,Selected_ML,48,0,414,0.074347,0.093392,0.402195,0.137740,-0.047888,0.079209,0.027921,0.352503
2,63,F5_Plus_Macro,raw,XGB_d2,Selected_ML,48,0,372,0.061384,0.077799,0.283031,0.306180,-0.050227,0.058687,0.038418,0.654625


## 4. Primary 21-day permutation importance on 2024 validation

In [5]:
from sklearn.base import clone

s = selected.loc[selected["Horizon"].eq(21)].iloc[0]
d = load_horizon(21)

y = "Target_RV_21"
features = feature_lists(d, 21)[s["Feature_Set"]]

v = d.loc[
    d["Date"].between("2024-01-01", "2024-12-31")
].dropna(subset=[y]).copy()

first = v.index.min()
tr = d.iloc[:first-21].dropna(subset=[y]).copy()

model = candidate_models()[s["Model"]]

y_train = tr[y].astype(float).to_numpy()
if s["Transform"] == "log":
    y_train = np.log(np.maximum(y_train, 1e-8))

model.fit(tr[features], y_train)

base_pred = np.asarray(model.predict(v[features]), dtype=float)
if s["Transform"] == "log":
    base_pred = np.exp(base_pred)
base_pred = np.maximum(base_pred, 1e-8)

base_qlike = qlike(v[y], base_pred)

rng = np.random.default_rng(RS)
importance_rows = []

for c in features:
    increases = []

    for _ in range(20):
        xp = v[features].copy()
        xp[c] = rng.permutation(xp[c].to_numpy())

        p = np.asarray(model.predict(xp), dtype=float)
        if s["Transform"] == "log":
            p = np.exp(p)
        p = np.maximum(p, 1e-8)

        increases.append(
            qlike(v[y], p) - base_qlike
        )

    importance_rows.append({
        "Feature": c,
        "QLIKE_Increase_Mean": float(np.mean(increases)),
        "QLIKE_Increase_SD": float(np.std(increases)),
    })

importance = (
    pd.DataFrame(importance_rows)
    .sort_values("QLIKE_Increase_Mean", ascending=False)
)

importance.to_csv(TABLES / "04_permutation_importance_21d.csv", index=False)
display(importance.head(25))

,Feature,QLIKE_Increase_Mean,QLIKE_Increase_SD
32,VIX_Level,7.880919e-03,0.001711
19,Drawdown_63,6.368905e-03,0.000804
7,RV21_Change_5D,3.166396e-03,0.000867
43,AUS_UNEMPLOYMENT,2.558926e-03,0.000294
30,VIX_Return_21D,2.119314e-03,0.001112
37,GOLD_Return_21D,2.019849e-03,0.000808
14,Return_21D,1.767168e-03,0.000371
41,COPPER_RV21,1.567315e-03,0.000570
18,Drawdown_21,1.415011e-03,0.000560
16,Downside_Vol_21,1.322820e-03,0.000316


### Final interpretation rules

- **Do not select models using 2025+ test results.**
- The 21-day horizon remains primary even if 63-day test performance is stronger.
- A stage improvement means the *best available configuration within that stage* improved; it does not prove that one newly added feature caused the improvement.
- If F4/F5 deteriorate, report that market/commodity/macro additions did not generalise under this specification.
- Very large QLIKE values from raw linear models should be interpreted alongside `04_nonpositive_raw_prediction_audit.csv`; near-zero floored volatility forecasts are heavily penalised by QLIKE.